# Metadata creation

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from src.utils.helpers import parse_weight

In [ ]:
def create_metadata(dataset_dir: Path) -> pd.DataFrame:
    
    data_dir = dataset_dir / "raw"

    if not data_dir.exists():
        print(f"[ERROR] Could not find: {data_dir}")
        print("Make sure your images are in:  data/raw/<food_type>/<weight_folder>/")
        sys.exit(1)


    food_info = []
    skipped_folders = []  # folders we couldn't parse
    food_type_dirs = [d for d in data_dir.iterdir() if d.is_dir()]

    for food_type_dir in food_type_dirs:
        portion_dirs = [p for p in food_type_dir.iterdir() if p.is_dir()]
        
        for portion_dir in portion_dirs:
            image_files = [f for f in portion_dir.iterdir() if f.is_file()]

            weight = parse_weight(portion_dir.name)
            if weight is None:
                skipped_folders.append(str(portion_dir))
                continue
            
            for image_file in image_files:
                food_info.append({
                    "type": food_type_dir.name,
                    "weight": weight,
                    "path": image_file  
                })


    if skipped_folders:
        print(f"  Skipped {len(skipped_folders)} folders (no weight pattern in name):")
        for f in skipped_folders:
            print(f"    - {f}")

    return pd.DataFrame(food_info)



In [ ]:
df = create_metadata(Path("data"))


In [ ]:
df

In [ ]:
data_dir = Path("data/raw")


csv_path = data_dir.parent / "metadata.csv"
df.to_csv(csv_path, index=False)
print(f"Metadata saved to: {csv_path}")